## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here
import sys
sys.setrecursionlimit(1000000)
def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return
    size = data[0]
    val_a = data[1]
    val_b = data[2]
    perm = data[3:3 + size]
    ops = []
    def bit_low(num):
        return num & -num
    step_len = (val_a - val_b + size) % size
    step_len = bit_low(step_len)
    if step_len == 0:
        step_len = size
    def op_magic():
        ops.append(0)
        for pos in range(size):
            if perm[pos] == val_a:
                perm[pos] = val_b
            elif perm[pos] == val_b:
                perm[pos] = val_a
    def op_add(delta):
        delta %= size
        if delta == 0:
            return
        ops.append(delta)
        for pos in range(size):
            perm[pos] = (perm[pos] + delta) % size
    def op_xor(mask):
        if mask == 0:
            return
        ops.append(-mask)
        for pos in range(size):
            perm[pos] ^= mask
    def get_pair(left, right):
        diff = (right - left + size - step_len + size) % size
        first = 0
        second = 0
        jump = size // 2
        while jump >= 2 * step_len:
            if diff >= jump:
                diff -= jump
                second += jump // 2
            else:
                first += jump // 2
            jump //= 2
        first += size // 2
        first += left & (step_len - 1)
        second += left & (step_len - 1)
        return first, second
    def swap_pos(pos_c, pos_d):
        if (pos_c // step_len) % 2 == (pos_d // step_len) % 2:
            if (pos_c // step_len) % 2 == 0:
                mid = (pos_c & (step_len - 1)) + step_len
            else:
                mid = pos_c & (step_len - 1)
            swap_pos(pos_c, mid)
            swap_pos(pos_d, mid)
            swap_pos(pos_c, mid)
        else:
            base_a, base_b = get_pair(val_a, val_b)
            map_c, map_d = get_pair(pos_c, pos_d)
            op_add((map_c - pos_c + size) % size)
            op_xor(map_c ^ base_a)
            op_add((val_a - base_a + size) % size)
            op_magic()
            op_add((base_a - val_a + size) % size)
            op_xor(map_c ^ base_a)
            op_add((pos_c - map_c + size) % size)
    def build_perm(cur_arr, cur_len):
        if cur_len == 1:
            return True, []
        half = cur_len // 2
        even_arr = [cur_arr[idx * 2] // 2 for idx in range(half)]
        odd_arr = [cur_arr[idx * 2 + 1] // 2 for idx in range(half)]
        ok_even, even_ops = build_perm(even_arr, half)
        if not ok_even:
            return False, []
        ok_odd, odd_ops = build_perm(odd_arr, half)
        if not ok_odd:
            return False, []
        result = []
        if cur_arr[0] % 2:
            if cur_len == 2:
                result.append(1)
            else:
                result.append(-1)
        even_mask = 0
        for item in even_ops:
            if item > 0:
                result.append(-1)
                result.append(1)
            else:
                result.append(item * 2)
                even_mask ^= -item * 2
        if even_mask:
            result.append(-even_mask)
        odd_mask = 0
        for item in odd_ops:
            if item > 0:
                result.append(1)
                result.append(-1)
            else:
                result.append(item * 2)
                odd_mask ^= -item * 2
        if (odd_mask & half) != (even_mask & half):
            return False, []
        if even_mask >= half:
            even_mask -= half
        if odd_mask >= half:
            odd_mask -= half
        if even_mask != odd_mask:
            return False, []
        merged = []
        for item in result:
            if not merged:
                merged.append(item)
            else:
                if item < 0 and merged[-1] < 0:
                    merged[-1] = -((-merged[-1]) ^ (-item))
                    if merged[-1] == 0:
                        merged.pop()
                else:
                    merged.append(item)
        return True, merged
    if step_len > 1:
        base_arr = [perm[pos] & (step_len - 1) for pos in range(step_len)]
        ok, op_list = build_perm(base_arr, step_len)
        if not ok:
            print(-1)
            return
        for item in op_list:
            if item > 0:
                op_add(item)
            else:
                op_xor(-item)
    for start in range(step_len):
        group = []
        for pos in range(start, size, step_len):
            group.append(perm[pos])
        group.sort()
        ptr = 0
        valid = True
        for pos in range(start, size, step_len):
            if group[ptr] != pos:
                valid = False
                break
            ptr += 1
        if not valid:
            print(-1)
            return
        for pos in range(start, size, step_len):
            if perm[pos] != pos:
                swap_pos(pos, perm[pos])
    for pos in range(size):
        if perm[pos] != pos:
            print(-1)
            return
    output = [str(len(ops))]
    for item in ops:
        if item == 0:
            output.append("0")
        elif item < 0:
            output.append(f"1 {-item}")
        else:
            output.append(f"2 {item}")
    sys.stdout.write("\n".join(output))
if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
## add your code here
import sys
from collections import deque
LIMIT = 10 ** 18
def check_route(cnt, end_pos, max_jump, money, raw_points):
    cheapest = {}
    for place, fee in raw_points:
        if place <= 0 or place >= end_pos:
            continue
        if place not in cheapest:
            cheapest[place] = fee
        elif fee < cheapest[place]:
            cheapest[place] = fee
    point_list = [(0, 0)]
    for place in sorted(cheapest):
        point_list.append((place, cheapest[place]))
    point_list.append((end_pos, 0))
    total = len(point_list)
    cost_dp = [LIMIT] * total
    cost_dp[0] = 0
    mono = deque()
    mono.append(0)
    for right in range(1, total):
        now_place, now_fee = point_list[right]
        while mono and now_place - point_list[mono[0]][0] > max_jump:
            mono.popleft()
        if mono:
            cost_dp[right] = cost_dp[mono[0]] + now_fee
        if cost_dp[right] == LIMIT:
            continue
        while mono and cost_dp[mono[-1]] >= cost_dp[right]:
            mono.pop()
        mono.append(right)
    return cost_dp[-1] <= money
def solve():
    nums = list(map(int, sys.stdin.buffer.read().split()))
    pos = 0
    answer = []
    while pos + 4 <= len(nums):
        station_count = nums[pos]
        road_length = nums[pos + 1]
        longest_dist = nums[pos + 2]
        max_cost = nums[pos + 3]
        pos += 4
        station_data = []
        for i in range(station_count):
            station_data.append((nums[pos], nums[pos + 1]))
            pos += 2
        if check_route(station_count, road_length, longest_dist, max_cost, station_data):
            answer.append("Yes")
        else:
            answer.append("No")
    sys.stdout.write("\n".join(answer))
if __name__ == "__main__":
    solve()

## C 最长回文

In [ ]:
## add your code here
import sys
HASH_BASE = 911382323
HASH_MASK = (1 << 64) - 1
def get_pal_radius(text):
    length = len(text)
    odd = [0] * length
    left, right = 0, -1
    for center in range(length):
        if center > right:
            radius = 1
        else:
            radius = odd[left + right - center]
            bound = right - center + 1
            if bound < radius:
                radius = bound
        while center - radius >= 0 and center + radius < length and text[center - radius] == text[center + radius]:
            radius += 1
        odd[center] = radius
        if center + radius - 1 > right:
            left = center - radius + 1
            right = center + radius - 1
    even = [0] * length
    left, right = 0, -1
    for center in range(length):
        if center > right:
            radius = 0
        else:
            radius = even[left + right - center + 1]
            bound = right - center + 1
            if bound < radius:
                radius = bound
        while center - radius - 1 >= 0 and center + radius < length and text[center - radius - 1] == text[center + radius]:
            radius += 1
        even[center] = radius
        if center + radius - 1 > right:
            left = center - radius
            right = center + radius - 1
    return odd, even
def main():
    raw = sys.stdin.buffer.read().split()
    if not raw:
        return
    size = int(raw[0])
    str_a = raw[1]
    str_b = raw[2]
    odd_a, even_a = get_pal_radius(str_a)
    odd_b, even_b = get_pal_radius(str_b)
    best = 1
    for radius in odd_a:
        cur = (radius << 1) - 1
        if cur > best:
            best = cur
    for radius in even_a:
        cur = radius << 1
        if cur > best:
            best = cur
    for radius in odd_b:
        cur = (radius << 1) - 1
        if cur > best:
            best = cur
    for radius in even_b:
        cur = radius << 1
        if cur > best:
            best = cur
    rev_a = str_a[::-1]
    power = [1] * (size + 1)
    hash_rev_a = [0] * (size + 1)
    hash_b = [0] * (size + 1)
    for i in range(size):
        power[i + 1] = (power[i] * HASH_BASE) & HASH_MASK
    cur_hash = 0
    for i, ch in enumerate(rev_a):
        cur_hash = (cur_hash * HASH_BASE + ch) & HASH_MASK
        hash_rev_a[i + 1] = cur_hash
    cur_hash = 0
    for i, ch in enumerate(str_b):
        cur_hash = (cur_hash * HASH_BASE + ch) & HASH_MASK
        hash_b[i + 1] = cur_hash
    candidates = []
    def add_candidate(a_end, b_begin, middle_len):
        nonlocal best
        if a_end < 0 or b_begin >= size:
            return
        max_left = a_end + 1
        max_right = size - b_begin
        extend_limit = max_left if max_left < max_right else max_right
        if extend_limit <= 0:
            return
        possible_len = middle_len + (extend_limit << 1)
        if possible_len <= best:
            return
        rev_pos = size - 1 - a_end
        candidates.append((possible_len, rev_pos, b_begin, middle_len, extend_limit))
    for center, radius in enumerate(odd_a):
        add_candidate(center - radius, center + radius - 1, (radius << 1) - 1)
    for center, radius in enumerate(even_a):
        if radius:
            add_candidate(center - radius - 1, center + radius - 1, radius << 1)
    for center, radius in enumerate(odd_b):
        add_candidate(center - radius + 1, center + radius, (radius << 1) - 1)
    for center, radius in enumerate(even_b):
        if radius:
            add_candidate(center - radius, center + radius, radius << 1)
    for split in range(size):
        add_candidate(split, split, 0)
    candidates.sort(reverse=True)
    def is_same(rev_pos, b_pos, length):
        left_hash = (hash_rev_a[rev_pos + length] - hash_rev_a[rev_pos] * power[length]) & HASH_MASK
        right_hash = (hash_b[b_pos + length] - hash_b[b_pos] * power[length]) & HASH_MASK
        return left_hash == right_hash
    for possible_len, rev_pos, b_pos, middle_len, extend_limit in candidates:
        if possible_len <= best:
            break
        need = ((best - middle_len) >> 1) + 1
        if need > extend_limit:
            continue
        if rev_a[rev_pos] != str_b[b_pos]:
            continue
        if not is_same(rev_pos, b_pos, need):
            continue
        matched = need
        if matched < extend_limit:
            next_len = matched + 1
            if is_same(rev_pos, b_pos, next_len):
                matched = next_len
                jump = 2
                while matched + jump <= extend_limit and is_same(rev_pos, b_pos, matched + jump):
                    matched += jump
                    jump <<= 1
                high = matched + jump - 1
                if high > extend_limit:
                    high = extend_limit
                while matched < high:
                    mid = (matched + high + 1) >> 1
                    if is_same(rev_pos, b_pos, mid):
                        matched = mid
                    else:
                        high = mid - 1
        cur_ans = middle_len + (matched << 1)
        if cur_ans > best:
            best = cur_ans
    print(best)
if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
## add your code here
import sys
MAXX = 100000 + 5
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 2)
    def add(self, idx, val):
        n = self.n
        tree = self.tree
        while idx <= n:
            tree[idx] += val
            idx += idx & -idx
    def sum(self, idx):
        tree = self.tree
        res = 0
        while idx > 0:
            res += tree[idx]
            idx -= idx & -idx
        return res
    def kth(self, k):
        tree = self.tree
        n = self.n
        idx = 0
        bit = 1
        while (bit << 1) <= n:
            bit <<= 1
        while bit:
            nxt = idx + bit
            if nxt <= n and tree[nxt] < k:
                idx = nxt
                k -= tree[nxt]
            bit >>= 1
        return idx + 1
    def first_at_least(self, pos):
        if pos < 1:
            pos = 1
        before = self.sum(pos - 1)
        total = self.sum(self.n)
        if total - before <= 0:
            return -1
        return self.kth(before + 1)
def solve_one(m, data, idx):
    cnt = [0] * MAXX
    lastI = [0] * MAXX
    lastO = [0] * MAXX
    bit = Fenwick(m)
    ans = -1
    for i in range(1, m + 1):
        op = data[idx]
        idx += 1
        if op != b'I' and op != b'O':
            if ans == -1:
                bit.add(i, 1)
            continue
        x = int(data[idx])
        idx += 1
        if ans != -1:
            continue
        if op == b'I':
            cnt[x] += 1
            if cnt[x] >= 2:
                pos = bit.first_at_least(lastI[x] + 1)
                if pos == -1:
                    ans = i
                else:
                    bit.add(pos, -1)
                    cnt[x] -= 1
            lastI[x] = i
        else:
            cnt[x] -= 1
            if cnt[x] < 0:
                pos = bit.first_at_least(lastO[x] + 1)
                if pos == -1:
                    ans = i
                else:
                    bit.add(pos, -1)
                    cnt[x] += 1
            lastO[x] = i
    return ans, idx
def main():
    data = sys.stdin.buffer.read().split()
    idx = 0
    res = []
    while idx < len(data):
        m = int(data[idx])
        idx += 1
        ans, idx = solve_one(m, data, idx)
        res.append(str(ans))
    sys.stdout.write("\n".join(res))
if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
## add your code here
import sys
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
    def root(self, node):
        while self.parent[node] != node:
            self.parent[node] = self.parent[self.parent[node]]
            node = self.parent[node]
        return node
    def merge(self, u, v):
        ru = self.root(u)
        rv = self.root(v)
        if ru != rv:
            self.parent[ru] = rv
def main():
    nums = list(map(int, sys.stdin.buffer.read().split()))
    if not nums:
        return
    count = nums[0]
    uf = UnionFind(count)
    same_x = {}
    same_y = {}
    p = 1
    for i in range(count):
        cx = nums[p]
        cy = nums[p + 1]
        p += 2
        if cx in same_x:
            uf.merge(i, same_x[cx])
        else:
            same_x[cx] = i
        if cy in same_y:
            uf.merge(i, same_y[cy])
        else:
            same_y[cy] = i
    groups = set()
    for i in range(count):
        groups.add(uf.root(i))
    print(len(groups) - 1)
if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
## add your code here

import sys
ANY_LEN = ord('*')
ANY_ONE = ord('?')
class PatternPiece:
    def __init__(self, piece_bytes):
        self.raw = piece_bytes
        self.size = len(piece_bytes)
        self.fixed_parts = []
        scan = 0
        total = len(piece_bytes)
        while scan < total:
            if piece_bytes[scan] == ANY_ONE:
                scan += 1
                continue
            nxt = scan
            while nxt < total and piece_bytes[nxt] != ANY_ONE:
                nxt += 1
            self.fixed_parts.append((scan, piece_bytes[scan:nxt]))
            scan = nxt
def can_place(file_name, piece, begin, border):
    if begin < 0 or begin + piece.size > border:
        return False
    for delta, token in piece.fixed_parts:
        if not file_name.startswith(token, begin + delta):
            return False
    return True
def earliest_place(file_name, piece, low, high):
    span = piece.size
    last_begin = high - span
    if low > last_begin:
        return -1
    if not piece.fixed_parts:
        return low
    key_offset, key_word = min(piece.fixed_parts, key=lambda item: (file_name.count(item[1]), -len(item[1])))
    search_from = low + key_offset
    search_to = last_begin + key_offset + len(key_word)
    while True:
        hit = file_name.find(key_word, search_from, search_to)
        if hit == -1:
            return -1
        begin = hit - key_offset
        matched = True
        for delta, token in piece.fixed_parts:
            if delta == key_offset and token == key_word:
                continue
            if not file_name.startswith(token, begin + delta):
                matched = False
                break
        if matched:
            return begin
        search_from = hit + 1
def is_accepted(file_name, rule, pieces):
    name_len = len(file_name)
    piece_cnt = len(pieces)
    cur = 0
    limit = name_len
    left_id = 0
    right_id = piece_cnt
    if rule[0] != ANY_LEN:
        head = pieces[0]
        if not can_place(file_name, head, 0, name_len):
            return False
        cur = head.size
        left_id = 1
    if rule[-1] != ANY_LEN:
        tail = pieces[-1]
        tail_begin = name_len - tail.size
        if tail_begin < cur:
            return False
        if not can_place(file_name, tail, tail_begin, name_len):
            return False
        limit = tail_begin
        right_id = piece_cnt - 1
    for piece_id in range(left_id, right_id):
        now_piece = pieces[piece_id]
        if now_piece.size == 0:
            continue
        place = earliest_place(file_name, now_piece, cur, limit)
        if place == -1:
            return False
        cur = place + now_piece.size
    return True
def main():
    raw_lines = sys.stdin.buffer.read().splitlines()
    if not raw_lines:
        return
    rule = raw_lines[0].strip()
    rule = rule.replace("？".encode(), b"?").replace("＊".encode(), b"*")
    file_count = int(raw_lines[1])
    file_list = [raw_lines[pos + 2].strip() for pos in range(file_count)]
    split_rules = rule.split(b'*')
    pieces = [PatternPiece(item) for item in split_rules]
    result = []
    for file_name in file_list:
        if is_accepted(file_name, rule, pieces):
            result.append("YES")
        else:
            result.append("NO")
    sys.stdout.write("\n".join(result))
if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
## add your code here
import sys
def solve():
    items = sys.stdin.buffer.read().split()
    if not items:
        return
    disk_count = int(items[0])
    move_order = items[1:7]
    priority = [[9] * 3 for _ in range(3)]
    for order_id, move in enumerate(move_order):
        source = move[0] - 65
        target = move[1] - 65
        priority[source][target] = order_id
    arrive = [[0] * 3 for _ in range(disk_count + 1)]
    step_num = [[0] * 3 for _ in range(disk_count + 1)]
    if disk_count == 0:
        print(0)
        return
    for start_col in range(3):
        best_col = -1
        best_order = 9
        for next_col in range(3):
            if next_col == start_col:
                continue
            if priority[start_col][next_col] < best_order:
                best_order = priority[start_col][next_col]
                best_col = next_col
        arrive[1][start_col] = best_col
        step_num[1][start_col] = 1
    for disk_id in range(2, disk_count + 1):
        for start_col in range(3):
            small_pos = start_col
            big_pos = start_col
            used_steps = 0
            while True:
                before = small_pos
                small_pos = arrive[disk_id - 1][before]
                used_steps += step_num[disk_id - 1][before]
                if small_pos == big_pos:
                    arrive[disk_id][start_col] = big_pos
                    step_num[disk_id][start_col] = used_steps
                    break
                big_pos = 3 - big_pos - small_pos
                used_steps += 1
    print(step_num[disk_count][0])
if __name__ == "__main__":
    solve()

## H 马步距离

In [ ]:
## add your code here
import sys
def get_min_steps(dx, dy):
    if dx < dy:
        dx, dy = dy, dx
    special = {
        (0, 0): 0,
        (1, 0): 3,
        (2, 2): 4
    }
    if (dx, dy) in special:
        return special[(dx, dy)]
    lower1 = (dx + 1) // 2
    lower2 = (dx + dy + 2) // 3
    step = lower1 if lower1 > lower2 else lower2
    if (step + dx + dy) & 1:
        step += 1
    return step
def solve():
    values = list(map(int, sys.stdin.buffer.read().split()))
    if not values:
        return
    start_x, start_y, end_x, end_y = values
    gap_x = abs(start_x - end_x)
    gap_y = abs(start_y - end_y)
    print(get_min_steps(gap_x, gap_y))
if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
## add your code here
class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        arr = [0] + heights + [0]
        mono = [0]
        best = 0
        for right in range(1, len(arr)):
            while arr[right] < arr[mono[-1]]:
                mid = mono.pop()
                left = mono[-1]
                area = arr[mid] * (right - left - 1)
                if area > best:
                    best = area
            mono.append(right)
        return best

## J 消防局的设立

In [ ]:
## add your code here
import sys
from collections import deque
def main():
    raw = sys.stdin.buffer.read().split()
    if not raw:
        return
    total_node = int(raw[0])
    edge = [[] for _ in range(total_node + 1)]
    pos = 1
    for node in range(2, total_node + 1):
        pre = int(raw[pos])
        pos += 1
        edge[node].append(pre)
        edge[pre].append(node)
    fa = [0] * (total_node + 1)
    dep = [0] * (total_node + 1)
    que = deque([1])
    visit_order = []
    fa[1] = 0
    dep[1] = 0
    while que:
        now = que.popleft()
        visit_order.append(now)
        for nxt in edge[now]:
            if nxt == fa[now]:
                continue
            fa[nxt] = now
            dep[nxt] = dep[now] + 1
            que.append(nxt)
    visit_order.reverse()
    placed = [False] * (total_node + 1)
    son_cover = [0] * (total_node + 1)
    grandson_cover = [0] * (total_node + 1)
    res = 0
    for cur in visit_order:
        father = fa[cur]
        grand = fa[father] if father else 0
        ok = False
        if placed[cur]:
            ok = True
        elif father and placed[father]:
            ok = True
        elif grand and placed[grand]:
            ok = True
        elif son_cover[cur] > 0:
            ok = True
        elif grandson_cover[cur] > 0:
            ok = True
        elif father and son_cover[father] > 0:
            ok = True
        if not ok:
            if grand:
                choose = grand
            elif father:
                choose = father
            else:
                choose = cur
            if not placed[choose]:
                placed[choose] = True
                choose_fa = fa[choose]
                choose_gfa = fa[choose_fa] if choose_fa else 0
                if choose_fa:
                    son_cover[choose_fa] += 1
                if choose_gfa:
                    grandson_cover[choose_gfa] += 1
            res += 1
    print(res)
if __name__ == "__main__":
    main()
